# developing

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import time
import numpy as np
import tensorflow as tf
from sklearn.model_selection import KFold
import random
import joblib
import gc
from tqdm import tqdm

2026-04-28 03:34:06.100712: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777347246.113719      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777347246.117914      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 03:34:06.134332: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

# variables

In [2]:
target_cols =['seizure_vote', 'lpd_vote','gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
test_path = "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
test_eegs = "/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/"
test_spectrograms = "/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/"

# confirm train_data 

In [3]:
ds_test = pd.read_csv(test_path)
ds_test.head(100)

,spectrogram_id,eeg_id,patient_id
0,2207717,2578018731,34153
1,2207717,2578018731,34153
2,2207717,2578018731,34153
3,2207717,2578018731,34153
4,8352559,1706196508,37552
...,...,...,...
95,26635579,1735082541,26091
96,26635579,1735082541,26091
97,26635579,2955034452,26091
98,26635579,2955034452,26091


# model

In [4]:
def positional_encoding(length, depth):
  depth = depth/2

  positions = np.arange(length)[:, np.newaxis]     # (seq, 1)
  depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth)

  angle_rates = 1 / (10000**depths)         # (1, depth)
  angle_rads = positions * angle_rates      # (pos, depth)

  pos_encoding = np.concatenate(
      [np.sin(angle_rads), np.cos(angle_rads)],
      axis=-1) 

  return tf.cast(pos_encoding, dtype=tf.float32)
                 
class PositionalEmbedding(tf.keras.layers.Layer):
  def __init__(self, d_model):
    super().__init__()
    self.d_model = d_model 
    self.pos_encoding = positional_encoding(length=2048, depth=d_model)

  def call(self, x):
    length = tf.shape(x)[1]
    # This factor sets the relative scale of the embedding and positonal_encoding.
    #x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x = x + self.pos_encoding[tf.newaxis, :length, :]
    return x

In [5]:
class BaseAttention(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

In [6]:
class GlobalSelfAttention(BaseAttention):
  def call(self, x):
    attn_output = self.mha(
        query=x,
        value=x,
        key=x)
    x = self.add([x, attn_output])
    x = self.layernorm(x)
    return x

In [7]:
class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dff, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x) 
    return x

In [8]:
class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()

    self.self_attention = GlobalSelfAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x):
    x = self.self_attention(x)
    x = self.ffn(x)
    return x

In [9]:
class RnnModel(tf.keras.Model):
  def __init__(self, *, num_layers, d_model, num_heads,
               dff, dropout_rate=0.1):
    super().__init__()

    self.d_model = d_model
    self.num_layers = num_layers
    #self.cov11 = tf.keras.layers.Conv1D(20, 1, activation='relu')
    #self.cov21 = tf.keras.layers.Conv1D(200, 1, activation='relu')
    #self.cov22 = tf.keras.layers.Conv1D(100, 1, activation='relu')
    
    self.pos_embedding = PositionalEmbedding(d_model=d_model)
    
    self.enc_layers = [
        EncoderLayer(d_model=d_model,
                     num_heads=num_heads,
                     dff=dff,
                     dropout_rate=dropout_rate)
        for _ in range(num_layers)]
    self.dropout = tf.keras.layers.Dropout(dropout_rate)
    self.layer1_100 = tf.keras.layers.Dense(108, activation='relu')
    self.layer1_6 = tf.keras.layers.Dense(6,activation='relu')
    self.add = tf.keras.layers.Add()
  def call(self, X):
    x1 = X['x1']
    x2 = X['x2']
    #print("x1",x1.shape)
    #print("x2",x2.shape)
    #x1 = self.cov11(x1)
    #x2 = self.cov21(x2)
    #x2 = self.cov22(x2)
    #print("x1",x1.shape)
    #print("x2",x2.shape)
    
    x = tf.concat([x1,x2],2)
    x = self.pos_embedding(x)  # Shape `(batch_size, seq_len, d_model)`.
    # Add dropout.
    x = self.dropout(x)
    for i in range(self.num_layers):
        x = self.enc_layers[i](x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = self.layer1_100(x)   
    x = self.layer1_6(x) 
    x = tf.keras.layers.Softmax(-1)(x)
    return x

# prediction

In [10]:
def loss_fn(labels, targets):
    loss = tf.math.abs(labels - targets)
    #loss = tf.math.reduce_mean(loss,1)
    #loss = tf.math.reduce_mean(loss,0)
    loss = tf.math.reduce_mean(loss)
    return loss

In [11]:

models_infor =[[0,160],[0,140],[0,120],[0,100],[0,80],[0,60],
               #[1,78],
               #[2,78],
               #[3,78],
               #[4,78],[4,74],
               #[5,78],[5,72],
               #[6,78]
              ]

In [12]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect(tpu="local") # "local" for 1VM TPU
    print('Running on TPU ')#, tpu.cluster_spec().as_dict()['worker'])
except:
    tpu = None
if tpu:
    strategy = tf.distribute.TPUStrategy(tpu)
    print("on TPU")
    print("REPLICAS: ", strategy.num_replicas_in_sync)
else:
    print("on GPU")
    strategy = tf.distribute.get_strategy()
model_list = []
with strategy.scope():
    for m_i in models_infor:
        model = RnnModel(num_layers=3,d_model=420,num_heads=2,dff=400,dropout_rate=0.5)
        model.build(input_shape = {"x1":[1,300,20],"x2":[1,300,400]})
        model.load_weights(f'/kaggle/input/hms-train-model-tf-transformer/{m_i[0]}_weights/model_epoch_{m_i[1]}.weights.h5')
        model_list.append(model)

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local
on GPU


2026-04-28 03:34:13.164802: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'rnn_model', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/kaggle/input/hms-train-model-tf-transformer/0_weights/model_epoch_160.weights.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [13]:
def prediction_re(ds):
    predictins = 0
    for model_p in model_list:
        predictins += model_p.predict(ds)
    predictins = predictins/len(model_list)
    return predictins

In [14]:
span = 1999
eps = 1e-6
n = (ds_test.shape[0]//span)+1
for i in range(n):
    temp_ds = ds_test.iloc[i*span:(i+1)*span,:].reset_index()
    print(temp_ds)
    list_x1 = []
    list_x2 = []
    for ii in range(temp_ds.shape[0]):
        eeg_id = temp_ds.loc[ii,"eeg_id"]
        min = 0
        values_eeg = pd.read_parquet(test_eegs+str(eeg_id)+".parquet").fillna(0).values[min:min+300,:]
        spectrogram_id = temp_ds.loc[ii,"spectrogram_id"]
        values_spc = pd.read_parquet(test_spectrograms+str(spectrogram_id)+".parquet").fillna(-1).values[min:min+300,1:]

        values_eeg=np.clip(values_eeg,np.exp(-6),np.exp(10))#最大值为89209464.0
        x1= np.log(values_eeg)#对数变换
        # Normalize the data
        data_mean = x1.mean(axis=(0, 1))
        data_std = x1.std(axis=(0, 1))
        x1 = (x1 - data_mean) / (data_std + eps)

        values_spc=np.clip(values_spc,np.exp(-6),np.exp(10))#最大值为89209464.0
        x2= np.log(values_spc)#对数变换
        # Normalize the data
        data_mean = x2.mean(axis=(0, 1))
        data_std = x2.std(axis=(0, 1))
        x2 = (x2 - data_mean) / (data_std + eps)
        list_x1.append(x1)
        list_x2.append(x2)
    pre_data = {'x1':np.array(list_x1),'x2':np.array(list_x2)}
    if i== 0:
        predictins = prediction_re(pre_data)
    else:
        predictins = np.concatenate([predictins,prediction_re(pre_data)])

      index  spectrogram_id      eeg_id  patient_id
0         0         2207717  2578018731       34153
1         1         2207717  2578018731       34153
2         2         2207717  2578018731       34153
3         3         2207717  2578018731       34153
4         4         8352559  1706196508       37552
...     ...             ...         ...         ...
1994   1994       457600492  3255376870       49358
1995   1995       457600492  3255376870       49358
1996   1996       457600492  3255376870       49358
1997   1997       457600492  3255376870       49358
1998   1998       457600492  3255376870       49358

[1999 rows x 4 columns]


ZeroDivisionError: division by zero

In [15]:
predic_ds = ds_test[["eeg_id"]]
for i in range(len(target_cols)):
    print(target_cols[i])
    predic_ds[target_cols[i]] = list(predictins[:,i])

seizure_vote


NameError: name 'predictins' is not defined

In [16]:

#sample_submission.csv
predic_ds.to_csv("submission.csv",index = None)

In [17]:
pd.read_csv("submission.csv")

,eeg_id
0,2578018731
1,2578018731
2,2578018731
3,2578018731
4,1706196508
...,...
9845,3349371726
9846,641615525
9847,641615525
9848,641615525
